### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


c:\Users\suhas\RAG-Proejct\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Read all the PDF's inside the directory

def process_all_pdfs(pdf_directory):
    # Process all PDF files in a directory
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f" Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 4 PDF files to process

Processing: attention.pdf
 Loaded 15 pages

Processing: embeddings.pdf
 Loaded 12 pages

Processing: objectdetection.pdf
 Loaded 6 pages

Processing: Proposal.pdf
 Loaded 5 pages

Total documents loaded: 38


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-03T00:07:29+00:00', 'author': '', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of T

In [4]:
# Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    # Split documents into smaller chunks for better RAG performance
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs


In [5]:
chunks = split_documents(all_pdf_documents)
chunks

Split 38 documents into 164 chunks

Example chunk:
Content: Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
...
Metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-03T00:07:29+00:00', 'author': '', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-03T00:07:29+00:00', 'author': '', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of T

### Embeddings

In [6]:
import numpy as np
import unicodedata
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def _clean_text(self, t) -> str:
        """
        Normalize and strip a single text input.
        - Converts non-strings to string
        - Strips lone surrogates (e.g. \\ud835 from bad PDF parsing)
        - Normalizes unicode (NFKC)
        - Removes control, format, surrogate, private-use, and unassigned characters
        - Collapses all whitespace variants into single spaces
        """
        if not isinstance(t, str):
            t = str(t)

        # Strip lone surrogates FIRST (unicodedata crashes on these)
        t = t.encode("utf-8", errors="ignore").decode("utf-8", errors="ignore")

        # Normalize unicode
        t = unicodedata.normalize("NFKC", t)

        # Remove all non-printable/control/format/surrogate characters
        t = "".join(c for c in t if unicodedata.category(c) not in {
            "Cc",   # control chars (\x00–\x1f, \x7f–\x9f)
            "Cf",   # format chars (zero-width spaces, soft hyphens, etc.)
            "Cs",   # surrogates
            "Co",   # private use
            "Cn",   # unassigned
        })

        # Collapse all whitespace variants into single spaces
        t = " ".join(t.split())

        return t.strip()
      
      
      
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        # Clean and filter all inputs
        cleaned_texts = [self._clean_text(t) for t in texts]
        cleaned_texts = [t for t in cleaned_texts if t]  # drop empty strings

        skipped = len(texts) - len(cleaned_texts)
        if skipped > 0:
            print(f"Warning: Skipped {skipped} invalid/empty text(s) out of {len(texts)}")

        if not cleaned_texts:
            raise ValueError("No valid texts to embed after cleaning (all inputs were None or empty)")

        # Encode one-by-one to identify any remaining bad chunks
        print(f"Validating {len(cleaned_texts)} texts before encoding...")
        for i, t in enumerate(cleaned_texts):
            try:
                self.model.encode([t])
            except Exception as e:
                print(f"Failed at index {i}: {repr(t[:200])}")
                raise RuntimeError(f"Bad text at index {i}: {repr(t[:200])}") from e

        print(f"Generating embeddings for {len(cleaned_texts)} texts...")
        embeddings = self.model.encode(cleaned_texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
      
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3337.20it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\suhas\AppData\Local\Temp\ipykernel_18128\1678534165.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### Vector DB

In [8]:
class VectorStore:
  # Manages document embeddings in a chromaDB vector store
  
  def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
    """
    Initialize the vector store
    
    Args:
        collection_name: Name of the ChromaDB collection
        persist_directory: Directory to persist the vector store
    """
    self.collection_name = collection_name
    self.persist_directory = persist_directory
    self.client = None
    self.collection = None
    self._initialize_store()
    
  def _initialize_store(self):
    # Initialize ChromaDB client
    try:
      # Create persistent ChromaDB client
      os.makedirs(self.persist_directory, exist_ok = True)
      self.client = chromadb.PersistentClient(path=self.persist_directory)
      
      #Get or create collection
      self.collection = self.client.get_or_create_collection(
        name = self.collection_name,
        metadata = {"description": "PDF document embeddings for RAG"}
      )
      print(f"Vector store initialized. Collection: {self.collection_name}")
      print(f"Existing documents in collection: {self.collection.count()}")
      
    except Exception as e:
      print(f"Error initializing vector store: {e}")
      raise
    
    
  def add_documents(self, documents: List[Any], embeddings: np.ndarray):
    """
    Add documents and their embeddings to the vector store
    
    Args:
        documents: List of Langchain documents
        embeddings: Corresponding embeddings for the documents

    """
    if len(documents) != len(embeddings):
      raise ValueError("Number of documents must match number of embeddings")
        
    print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
    ids = []
    metadatas = []
    documents_text = []
    embeddings_list = []
        
    for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
      # Generate unique ID
      doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
      ids.append(doc_id)
            
      # Prepare metadata
      metadata = dict(doc.metadata)
      metadata['doc_index'] = i
      metadata['content_length'] = len(doc.page_content)
      metadatas.append(metadata)
            
      # Document content
      documents_text.append(doc.page_content)
            
      # Embedding
      embeddings_list.append(embedding.tolist())
        
    # Add to collection
    try:
      self.collection.add(
        ids=ids,
        embeddings=embeddings_list,
        metadatas=metadatas,
        documents=documents_text
      )
      print(f"Successfully added {len(documents)} documents to vector store")
      print(f"Total documents in collection: {self.collection.count()}")
            
    except Exception as e:
      print(f"Error adding documents to vector store: {e}")
      raise

vectorstore = VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 164


In [9]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-03T00:07:29+00:00', 'author': '', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of T

In [10]:
# Debug your chunks BEFORE passing to generate_embeddings
texts = [doc.page_content for doc in chunks]

print(f"Total chunks: {len(texts)}")
for i, t in enumerate(texts):
    if not isinstance(t, str) or not t.strip():
        print(f"  BAD chunk at index {i}: type={type(t)}, value={repr(t)}")

print("Done inspecting.")

Total chunks: 164
Done inspecting.


In [11]:
# Convert text to embeddings
texts = [doc.page_content for doc in chunks]

# Clean the chunks' page_content in-place before storing
for doc in chunks:
    doc.page_content = embedding_manager._clean_text(doc.page_content)

# Generate embeddings from cleaned texts
texts = [doc.page_content for doc in chunks]
embeddings = embedding_manager.generate_embeddings(texts)

# Store in vector database
vectorstore.add_documents(chunks, embeddings)

Validating 164 texts before encoding...
Generating embeddings for 164 texts...


Batches: 100%|██████████| 6/6 [00:09<00:00,  1.59s/it]


Generated embeddings with shape: (164, 384)
Adding 164 documents to vector store...
Successfully added 164 documents to vector store
Total documents in collection: 328


### Retriever Pipeline From VectorStore

In [12]:
class RAGRetriever:
    # Handles query-based retrieval from the vector store
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever = RAGRetriever(vectorstore, embedding_manager)



In [13]:
rag_retriever

In [14]:
rag_retriever.retrieve("What is attention is all you need")

Retrieving documents for query: 'What is attention is all you need'
Top K: 5, Score threshold: 0.0
Validating 1 texts before encoding...
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 58.58it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_b98d97f5_12',
  'content': '3.2 AttentionAn attention function can be described as mapping a query and a set of key-value pairs to an output,where the query, keys, values, and output are all vectors. The output is computed as a weighted sum3',
  'metadata': {'creationdate': '2023-08-03T00:07:29+00:00',
   'trapped': '/False',
   'creator': 'LaTeX with hyperref',
   'title': '',
   'subject': '',
   'producer': 'pdfTeX-1.40.25',
   'source': '..\\data\\pdf\\attention.pdf',
   'moddate': '2023-08-03T00:07:29+00:00',
   'author': '',
   'total_pages': 15,
   'source_file': 'attention.pdf',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'doc_index': 12,
   'keywords': '',
   'file_type': 'pdf',
   'page': 2,
   'page_label': '3',
   'content_length': 213},
  'similarity_score': 0.09521329402923584,
  'distance': 0.9047867059707642,
  'rank': 1},
 {'id': 'doc_9c956bb0_12',
  'content': '3.2 AttentionAn attentio

In [15]:
rag_retriever.retrieve("Recurrent Neural Net Language Model (RNNLM)")


Retrieving documents for query: 'Recurrent Neural Net Language Model (RNNLM)'
Top K: 5, Score threshold: 0.0
Validating 1 texts before encoding...
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 42.44it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_4c83bba6_64',
  'content': 'would require log2(V ) outputs to be evaluated, the Huffman tree based hierarchical softmax requiresonly about log2(U nigram perplexity(V )). For example when the vocabulary size is one millionwords, this results in about two times speedup in evaluation. While this is not crucial speedup forneural network LMs as the computational bottleneck is in theN ×D ×H term, we will later proposearchitectures that do not have hidden layers and thus depend heavily on the efficiency of the softmaxnormalization.2.2 Recurrent Neural Net Language Model (RNNLM)Recurrent neural network based language model has been proposed to overcome certain limitationsof the feedforward NNLM, such as the need to specify the context length (the order of the modelN),and because theoretically RNNs can efficiently represent more complex patterns than the shallowneural networks [15, 2]. The RNN model does not have a projection layer; only input, hidden and',
  'metadata': {'creation

In [16]:
rag_retriever.retrieve("Examples of the Learned Relationships")


Retrieving documents for query: 'Examples of the Learned Relationships'
Top K: 5, Score threshold: 0.0
Validating 1 texts before encoding...
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 40.83it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_01b6dbc0_91',
  'content': 'leads to a new state of the art result 58.9% accuracy (59.2% on the development part of the set and58.7% on the test part of the set).5 Examples of the Learned RelationshipsTable 8 shows words that follow various relationships. We follow the approach described above: therelationship is defined by subtracting two word vectors, and the result is added to another word. Thusfor example, Paris - France + Italy = Rome . As it can be seen, accuracy is quite good, althoughthere is clearly a lot of room for further improvements (note that using our accuracy metric that9',
  'metadata': {'total_pages': 12,
   'trapped': '/False',
   'file_type': 'pdf',
   'creator': 'LaTeX with hyperref package',
   'page_label': '9',
   'source': '..\\data\\pdf\\embeddings.pdf',
   'creationdate': '2013-09-10T00:03:46+00:00',
   'keywords': '',
   'moddate': '2013-09-10T00:03:46+00:00',
   'source_file': 'embeddings.pdf',
   'content_length': 567,
   'producer': 'pdfTeX-

### RAG Pipeline- VectorDB To LLM Output Generation

In [17]:
import os
from dotenv import load_dotenv
load_dotenv()


True

In [18]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [19]:
class GroqLLM:
    def __init__(self, model_name: str = "gemma2-9b-it", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"
    


In [20]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: gemma2-9b-it
Groq LLM initialized successfully!


In [21]:
# Get the context from the retriever and pass it to the LLM

rag_retriever.retrieve("Recurrent Neural Net Language Model (RNNLM)")

Retrieving documents for query: 'Recurrent Neural Net Language Model (RNNLM)'
Top K: 5, Score threshold: 0.0
Validating 1 texts before encoding...
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 48.94it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_4c83bba6_64',
  'content': 'would require log2(V ) outputs to be evaluated, the Huffman tree based hierarchical softmax requiresonly about log2(U nigram perplexity(V )). For example when the vocabulary size is one millionwords, this results in about two times speedup in evaluation. While this is not crucial speedup forneural network LMs as the computational bottleneck is in theN ×D ×H term, we will later proposearchitectures that do not have hidden layers and thus depend heavily on the efficiency of the softmaxnormalization.2.2 Recurrent Neural Net Language Model (RNNLM)Recurrent neural network based language model has been proposed to overcome certain limitationsof the feedforward NNLM, such as the need to specify the context length (the order of the modelN),and because theoretically RNNs can efficiently represent more complex patterns than the shallowneural networks [15, 2]. The RNN model does not have a projection layer; only input, hidden and',
  'metadata': {'trapped'

### Integration Vectordb Context pipeline With LLM output

In [22]:
# Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

# Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0.1,
    max_tokens=1024
)

# 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    # retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    # generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [23]:
answer=rag_simple("What is attention mechanism?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is attention mechanism?'
Top K: 3, Score threshold: 0.0
Validating 1 texts before encoding...
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 36.09it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


The attention mechanism is a technique used in deep learning models, such as transformers, to focus on specific parts of the input data when processing it. It allows the model to weigh the importance of different input elements and concentrate on the most relevant ones, enabling it to capture long-distance dependencies and relationships between words in a sentence.


### Enhanced RAG Pipeline Features

In [32]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Research on the Online Update Method for Retrieval- Augmented Generation (RAG) Model with Incremental Learning", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Research on the Online Update Method for Retrieval- Augmented Generation (RAG) Model with Incremental Learning'
Top K: 3, Score threshold: 0.1
Validating 1 texts before encoding...
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 36.54it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: The research proposes an innovative online update method combined with an incremental learning mechanism to address the challenges of online update and knowledge integration of the Retrieval-Augmented Generation (RAG) model in a dynamic environment.
Sources: [{'source': 'Proposal.pdf', 'page': 3, 'score': 0.6189348101615906, 'preview': 'Figure 3. Consistency Score Comparison Across MethodsV. ConclusionsIn conclusion, an innovative online update methodcombined with an incremental learning mechanism is proposedto address the challenges of online update and knowledgeintegration of retrieval-augmented generation (RAG) model ina dynamic...'}, {'source': 'Proposal.pdf', 'page': 3, 'score': 0.6189348101615906, 'preview': 'Figure 3. Consistency Score Comparison Across MethodsV. ConclusionsIn conclusion, an innovative online update methodcombined with an incremental learning mechanism is proposedto address the challenges of online update and knowledgeintegration of retrieval-augmented g

In [35]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is Efficient Estimation of Word Representations in Vector Space", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is Efficient Estimation of Word Representations in Vector Space'
Top K: 3, Score threshold: 0.1
Validating 1 texts before encoding...
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 50.27it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
Efficient E

stimation of Word Representations inVector SpaceTomas MikolovGoogle Inc., Mountain View, CAtmikolov@google.comKai ChenGoogle Inc., Mountain View, CAkaichen@google.comGreg CorradoGoogle Inc., Mountain View, CAgcorrado@google.comJeffrey DeanG

oogle Inc., Mountain View, CAjeff@google.comAbstractWe propose two novel model architectures for computing continuous vector repre-sentations of words from very large data sets. The quality of these representationsis measured in a word similarity task, and the results are compared to the previ-ously best performing techniques based on different types of neural networks. Weobserve large improvements in accuracy at much lower computational cost, i.e. ittakes less than a day to learn high quality word vectors from a 1.6 billion wordsdata set. Furthermore, we show that these vectors provide state-of-the-art perfor-mance on our test set for measuring syntactic and semantic word similarities.1 Introduction

Efficient Estimation of Word Representations inVector SpaceTomas MikolovGoogle Inc., Mountain View, CAtmikolov@google.comKai ChenGoogle Inc., Mountain View, CAkaichen@google.comGreg CorradoGoogle Inc., Mountain View, CAgcorrado@google.comJeffrey DeanGoogle Inc., Mountain View, CAjeff@goog